# FreeCompute V1 - dual-T4 acceptance worker

Upload this notebook from `v2/safety-and-agentdriver-spike`.
The equivalent `universal_dual_gpu_server.ipynb` wrapper is kept in sync.
The root proof notebook is historical evidence, not this authenticated worker.

Settings: **GPU T4 x2**, **Internet ON**, **Persistence None**. No inputs required.
For Cloudflare, enable `FREECOMPUTE_API_KEY` in Kaggle Secrets. Code cell 1
defaults to a temporary Cloudflare URL. Tailscale remains an optional mode and
requires `TAILSCALE_AUTHKEY` only if selected. Never paste keys into source.

Use the numbers in the code-cell headings, excluding this Markdown introduction.
Run code cells **1 through 8**, one at a time. **Do not Run All**
or Save & Run All. Code cell 9 is guarded shutdown, only after acceptance finishes.
Do not rerun configuration/startup/transport while the local harness uses this worker.
Stop on any error. Do not change model, context, split, caches or engine flags.

Code cell 6 must report `SUPERVISOR HEALTHY`; code cell 7 prints the selected endpoint;
code cell 8 must report `WORKER READY FOR LOCAL ACCEPTANCE` and the observed GPUs.
Return the endpoint and code cell 8 manifest to Codex, with the bearer token stored
locally in the ignored `.env` as `FREECOMPUTE_API_KEY=...`. No local harness command
is required yet: Codex will configure the Stage 4 worker and run acceptance.

The 65,536 context setting preserves the historical profile. It is not fresh
long-context evidence. This notebook makes no model generation requests. Cloudflare Quick Tunnels
do not support SSE, so their URL cannot establish streaming acceptance.


In [ ]:
# ==============================================================================
# 1. MODEL SELECTION & RUNTIME CONFIGURATION
# Uncomment any preset below to switch models, or specify any Hugging Face GGUF!
# ==============================================================================
from kaggle_secrets import UserSecretsClient
from typing import Any
import re
import builtins

class SecretScrubber:
    DEFAULT_PATTERNS = [
        (re.compile(r"(Bearer\s+)[A-Za-z0-9_\-.]{8,}", re.I), r"\1[REDACTED_TOKEN]"),
        (re.compile(r"tskey-auth-[A-Za-z0-9_\-]+", re.I), "[REDACTED_TAILSCALE_KEY]"),
        (re.compile(r"sk-[A-Za-z0-9_\-]{8,}", re.I), "[REDACTED_SECRET_KEY]"),
        (re.compile(r"https?://[a-zA-Z0-9\-]+\.trycloudflare\.com", re.I), "https://[tunnel].trycloudflare.com"),
    ]

    def __init__(self):
        self._exact_secrets = set()

    def register_secret(self, secret):
        if isinstance(secret, str) and secret.strip():
            self._exact_secrets.add(secret.strip())

    def scrub(self, value: Any) -> str:
        text = "" if value is None else str(value)
        for secret in sorted(self._exact_secrets, key=len, reverse=True):
            text = text.replace(secret, "[REDACTED_SECRET]")
        for pattern, replacement in self.DEFAULT_PATTERNS:
            text = pattern.sub(replacement, text)
        return text

    def structured(self, value):
        if isinstance(value, dict):
            return {self.scrub(k): self.structured(v) for k, v in value.items()}
        if isinstance(value, (tuple, list)):
            return [self.structured(v) for v in value]
        return self.scrub(value) if isinstance(value, str) else value

scrubber = SecretScrubber()
def print(*values, **kwargs):
    builtins.print(*(scrubber.scrub(value) for value in values), **kwargs)

for name in ('llama_proc', 'supervisor_proc', 'ts_daemon', 'cf_proc'):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        raise RuntimeError('Worker already running; do not rerun configuration.')

# Choose the transport before retrieving its secrets.
TRANSPORT = 'cloudflare'  # Optional: 'tailscale' for private streaming acceptance.
if TRANSPORT not in ('cloudflare', 'tailscale'):
    raise ValueError('TRANSPORT must be cloudflare or tailscale.')
secret_client = UserSecretsClient()
try:
    bearer_key = secret_client.get_secret('FREECOMPUTE_API_KEY')
except Exception:
    raise RuntimeError('Enable FREECOMPUTE_API_KEY in Kaggle Secrets.') from None
if not bearer_key or not bearer_key.strip():
    raise ValueError('FREECOMPUTE_API_KEY must be nonempty.')
scrubber.register_secret(bearer_key)
tailscale_key = ''
if TRANSPORT == 'tailscale':
    try:
        tailscale_key = secret_client.get_secret('TAILSCALE_AUTHKEY')
    except Exception:
        raise RuntimeError('Enable TAILSCALE_AUTHKEY in Kaggle Secrets for Tailscale mode.') from None
    if not tailscale_key or not tailscale_key.strip():
        raise ValueError('TAILSCALE_AUTHKEY must be nonempty for Tailscale mode.')
    scrubber.register_secret(tailscale_key)

# ------------------------------------------------------------------------------
# PRESET 1: Qwen 3.8-27B Abliterated (DEFAULT)
# Uncensored / abliterated, strong reasoning and coding, 64K context.
# ------------------------------------------------------------------------------
MODEL_PRESET = {
    "REPO_ID": "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF",
    "FILENAME": "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf",
    "REVISION": "3f101cd22b7999228bbd5d79a33975414eb9758b",
    "MODEL_ALIAS": "qwen3.8-27b-huihui-abliterated-q4",
    "CTX_SIZE": 65536,
}

# ------------------------------------------------------------------------------
# PRESET 2: Qwen 2.5 Coder 32B Instruct
# Premier open coding model for deep architectural refactoring.
# ------------------------------------------------------------------------------
# MODEL_PRESET = {
#     "REPO_ID": "Qwen/Qwen2.5-Coder-32B-Instruct-GGUF",
#     "FILENAME": "qwen2.5-coder-32b-instruct-q4_k_m.gguf",
#     "REVISION": "main",
#     "MODEL_ALIAS": "qwen2.5-coder-32b",
#     "CTX_SIZE": 49152,
# }

# ------------------------------------------------------------------------------
# PRESET 3: DeepSeek-R1 Distill Qwen 32B
# Intense reasoning and mathematical problem solving.
# ------------------------------------------------------------------------------
# MODEL_PRESET = {
#     "REPO_ID": "unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF",
#     "FILENAME": "DeepSeek-R1-Distill-Qwen-32B-Q4_K_M.gguf",
#     "REVISION": "main",
#     "MODEL_ALIAS": "deepseek-r1-distill-32b",
#     "CTX_SIZE": 32768,
# }

# ------------------------------------------------------------------------------
# PRESET 4: Mistral Small 24B Instruct 2501
# Fast, low-latency, great conversational agent.
# ------------------------------------------------------------------------------
# MODEL_PRESET = {
#     "REPO_ID": "bartowski/mistral-small-24b-instruct-2501-GGUF",
#     "FILENAME": "mistral-small-24b-instruct-2501-Q4_K_M.gguf",
#     "REVISION": "main",
#     "MODEL_ALIAS": "mistral-small-24b",
#     "CTX_SIZE": 32768,
# }

CONFIG = {
    **MODEL_PRESET,
    "LLAMA_COMMIT": "2b129ccfa03aea330d2d9ac4650a10de393dbe3a",
    "USE_DATASET_CACHE": False,  # Acceptance builds/downloads the pinned assets.

    # Dual-GPU layer split settings (50/50 across both Tesla T4s)
    "N_GPU_LAYERS": 999,       # Offload all layers to GPUs
    "SPLIT_MODE": "layer",     # 'layer' splits layers across GPUs (CUDA0 + CUDA1)
    "TENSOR_SPLIT": "1,1",     # 50/50 layer balance across dual T4s
    "BATCH_SIZE": 512,
    "UBATCH_SIZE": 128,
    "CACHE_TYPE_K": "f16",
    "CACHE_TYPE_V": "f16",

    # Ports & Security Gateway
    "SUPERVISOR_PORT": 8081,
    "LLAMA_PORT": 8080,
    "API_KEY": bearer_key,  # Private Kaggle Secret; never printed

    # Cloudflare is a temporary URL for startup checks; streaming needs another route.
    "TRANSPORT": TRANSPORT,
    "TAILSCALE_AUTHKEY": tailscale_key,
}

print(f"Configured model : {CONFIG['MODEL_ALIAS']}")
print(f"Hugging Face repo: {CONFIG['REPO_ID']}")
print(f"Target context   : {CONFIG['CTX_SIZE']} tokens on dual-GPU {CONFIG['SPLIT_MODE']} split ({CONFIG['TENSOR_SPLIT']})")

In [ ]:
# ==============================================================================
# 2. HARDWARE & SCRATCH PREFLIGHT
# ==============================================================================
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path('/kaggle/working')
SCRATCH = Path('/kaggle/tmp/freecompute')
MODELS_DIR = SCRATCH / 'models'
BUILD_DIR = SCRATCH / 'llama.cpp' / 'build'

SCRATCH.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_home')

# Verify GPUs
smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free', '--format=csv,noheader'],
    text=True, capture_output=True, check=True
).stdout.strip()
print('Detected GPUs:\n' + smi)
rows = smi.splitlines()
if len(rows) != 2 or any('T4' not in row.split(',')[1] for row in rows):
    raise RuntimeError('This acceptance profile requires exactly two NVIDIA T4 GPUs.')

cuda_bin = Path('/usr/local/cuda/bin')
if cuda_bin.is_dir():
    os.environ['PATH'] = str(cuda_bin) + os.pathsep + os.environ.get('PATH', '')

for exe in ('git', 'cmake', 'nvcc'):
    if not shutil.which(exe):
        raise RuntimeError(f'Required executable {exe!r} is missing.')

free_scratch_gb = shutil.disk_usage(SCRATCH).free / (1024**3)
print(f'Free scratch disk: {free_scratch_gb:.1f} GiB')
if free_scratch_gb < 20:
    raise RuntimeError('Less than 20 GiB scratch space available.')
print('PREFLIGHT CHECKS PASSED.')

In [ ]:
# ==============================================================================
# 3. REUSABLE DATASET DETECTION (FAST START vs COLD BUILD)
# ==============================================================================
cached_server = None
cached_model = None

input_dir = Path('/kaggle/input')
if CONFIG['USE_DATASET_CACHE'] and input_dir.is_dir():
    for p in input_dir.glob('**/llama-server'):
        if p.is_file() and os.access(p, os.X_OK):
            cached_server = p
            break
    for p in input_dir.glob(f'**/{CONFIG["FILENAME"]}'):
        if p.is_file():
            cached_model = p
            break

SERVER_BIN = None
MODEL_PATH = None

if cached_server:
    print(f'FAST START: Found precompiled llama-server at {cached_server}')
    local_bin = SCRATCH / 'bin' / 'llama-server'
    local_bin.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached_server, local_bin)
    local_bin.chmod(0o755)
    SERVER_BIN = local_bin
else:
    print('COLD START: No precompiled llama-server in /kaggle/input; will build from source.')

if cached_model:
    print(f'FAST START: Found cached model checkpoint at {cached_model}')
    MODEL_PATH = cached_model
else:
    print(f'COLD START: Model {CONFIG["FILENAME"]} not in /kaggle/input; will download from Hugging Face.')

In [ ]:
# ==============================================================================
# 4. ENGINE BUILD (PINNED COMMIT 2b129cc WITH -DGGML_CUDA_NO_VMM=ON)
# ==============================================================================
if SERVER_BIN is None or not SERVER_BIN.is_file():
    SOURCE_DIR = SCRATCH / 'llama.cpp'
    if not SOURCE_DIR.exists():
        subprocess.run([
            'git', 'clone', '--no-checkout', '--depth', '1',
            'https://github.com/ggml-org/llama.cpp.git', str(SOURCE_DIR)
        ], check=True)
    
    subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', '--depth', '1', 'origin', CONFIG['LLAMA_COMMIT']], check=True)
    subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', CONFIG['LLAMA_COMMIT']], check=True)

    commit = subprocess.run(
        ['git', '-C', str(SOURCE_DIR), 'rev-parse', 'HEAD'],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    if commit != CONFIG['LLAMA_COMMIT']:
        raise RuntimeError('llama.cpp checkout does not match the acceptance pin.')
    print('Building llama.cpp commit:', commit)

    # Auto-detect CUDA compute capability (75 for T4, 80 for A100, 89 for L4)
    cuda_arch = "75"
    try:
        smi_cap = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            text=True
        ).splitlines()[0].strip().replace('.', '')
        if smi_cap:
            cuda_arch = smi_cap
    except Exception:
        pass
    print(f'Targeting CUDA architecture: sm_{cuda_arch}')

    BUILD = SOURCE_DIR / 'build'
    subprocess.run([
        'cmake',
        '-S', str(SOURCE_DIR),
        '-B', str(BUILD),
        '-DGGML_CUDA=ON',
        '-DGGML_CUDA_NO_VMM=ON',
        f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}',
        '-DCMAKE_BUILD_TYPE=Release',
        '-DLLAMA_BUILD_TESTS=OFF',
    ], check=True)

    print('Compiling llama-server with --parallel 2...')
    subprocess.run([
        'cmake', '--build', str(BUILD),
        '--target', 'llama-server',
        '--parallel', '2'
    ], check=True)

    SERVER_BIN = BUILD / 'bin' / 'llama-server'
    if not SERVER_BIN.is_file():
        raise RuntimeError('Build failed to produce llama-server binary.')
    print('ENGINE BUILD SUCCEEDED.')

subprocess.run([str(SERVER_BIN), '--version'], check=True)
print('llama-server ready at:', SERVER_BIN)

In [ ]:
# ==============================================================================
# 5. MODEL CHECKPOINT RETRIEVAL
# ==============================================================================
import sys
if MODEL_PATH is None or not Path(MODEL_PATH).is_file():
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'huggingface_hub>=0.34,<1.0', 'hf_xet>=1.0', 'requests>=2.28',
    ], check=True)
    from huggingface_hub import HfApi, hf_hub_download

    print(f'Downloading {CONFIG["FILENAME"]} from {CONFIG["REPO_ID"]}...')
    download_kwargs = {
        'repo_id': CONFIG['REPO_ID'],
        'filename': CONFIG['FILENAME'],
        'local_dir': str(MODELS_DIR),
    }
    if CONFIG.get('REVISION'):
        download_kwargs['revision'] = CONFIG['REVISION']

    downloaded = hf_hub_download(**download_kwargs)
    MODEL_PATH = Path(downloaded)
    size_gb = MODEL_PATH.stat().st_size / 1e9
    print(f'Downloaded {MODEL_PATH.name}: {size_gb:.2f} GB')

print('MODEL READY AT:', MODEL_PATH)

In [ ]:
# ==============================================================================
# 6. START LLAMA-SERVER & AUTHENTICATED SUPERVISOR
# ==============================================================================
import time
import requests

if not str(CONFIG.get('API_KEY', '')).strip():
    raise ValueError('Supervisor key must be nonempty before starting inference')

for name in ('llama_proc', 'supervisor_proc'):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        raise RuntimeError('Worker already running; do not rerun startup.')

LOG_PATH = WORK / 'llama_server.log'
SUPERVISOR_PY = SCRATCH / 'supervisor.py'

supervisor_script = '#!/usr/bin/env python3\n"""\nkaggle/supervisor.py — Lightweight Authenticated Supervisor & Watchdog for llama-server\nRuns inside the Kaggle GPU container, providing:\n1. Token-based authentication (Bearer token gate on all endpoints).\n2. Health & Telemetry endpoint (/health) with real container uptime and dual-GPU stats.\n3. Streaming OpenAI-compatible reverse proxy (/v1/chat/completions) to llama-server.\n4. Process lifecycle & watchdog to monitor, restart, and safely terminate llama-server.\n"""\n\nimport hmac\nimport re\nimport argparse\nimport http.server\nimport json\nimport os\nimport shutil\nimport socketserver\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\n\nclass SupervisorConfig:\n    def __init__(self, port=8081, llama_port=8080, api_key=None, llama_cmd=None, log_path=""):\n        self.port = port\n        self.llama_port = llama_port\n        self.api_key = os.environ.get("SUPERVISOR_API_KEY") if api_key is None else api_key\n        if not self.api_key or not self.api_key.strip():\n            raise ValueError("SUPERVISOR_API_KEY must be nonempty")\n        self.llama_cmd = llama_cmd or []\n        self.log_path = log_path or "/kaggle/working/llama_server.log"\n        self.start_time = time.time()\n        self.llama_process = None\n        self.watchdog_active = True\n\n\nconfig = None\n\ndef scrub(text):\n    text = str(text)\n    if config and config.api_key:\n        text = text.replace(config.api_key, "[REDACTED_SECRET]")\n    text = re.sub(r"(?i)Bearer\\s+\\S+", "Bearer [REDACTED_TOKEN]", text)\n    text = re.sub(r"https?://[^\\s]+", "[REDACTED_URL]", text)\n    return text\n\n\ndef get_container_uptime_seconds():\n    """Extract real container uptime from /proc/uptime if available."""\n    try:\n        with open("/proc/uptime", "r") as f:\n            return float(f.readline().split()[0])\n    except Exception:\n        return time.time() - config.start_time\n\n\ndef get_gpu_telemetry():\n    """Query nvidia-smi for dual Tesla T4 memory, temperature, and utilization."""\n    if not shutil.which("nvidia-smi"):\n        return []\n    try:\n        cmd = [\n            "nvidia-smi",\n            "--query-gpu=index,name,memory.used,memory.total,temperature.gpu,utilization.gpu",\n            "--format=csv,noheader,nounits",\n        ]\n        out = subprocess.check_output(cmd, text=True, stderr=subprocess.DEVNULL).strip()\n        gpus = []\n        for line in out.splitlines():\n            parts = [p.strip() for p in line.split(",")]\n            if len(parts) >= 6:\n                gpus.append({\n                    "index": int(parts[0]),\n                    "name": parts[1],\n                    "vramUsedMiB": int(parts[2]),\n                    "vramTotalMiB": int(parts[3]),\n                    "tempC": int(parts[4]),\n                    "utilizationPct": int(parts[5]),\n                })\n        return gpus\n    except Exception as exc:\n        return [{"error": scrub(exc)}]\n\n\ndef check_llama_health():\n    """Check if the internal llama-server answers on its loopback port."""\n    url = f"http://127.0.0.1:{config.llama_port}/health"\n    try:\n        req = urllib.request.Request(url, method="GET")\n        with urllib.request.urlopen(req, timeout=3) as resp:\n            return resp.status == 200\n    except Exception:\n        return False\n\n\ndef start_llama_server():\n    """Launch the llama-server subprocess if a command was configured."""\n    if not config.llama_cmd:\n        return None\n    print(f"[SUPERVISOR] Starting llama-server on port {config.llama_port}...")\n    log_dir = os.path.dirname(config.log_path)\n    if log_dir:\n        os.makedirs(log_dir, exist_ok=True)\n    log_file = open(config.log_path, "a", encoding="utf-8")\n    proc = subprocess.Popen(\n        config.llama_cmd,\n        stdout=log_file,\n        stderr=subprocess.STDOUT,\n    )\n    config.llama_process = proc\n    return proc\n\n\nclass SupervisorHandler(http.server.BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n\n    def log_message(self, format, *args):\n        sys.stderr.write(scrub(f"[SUPERVISOR {self.address_string()}] {format % args}") + "\\n")\n\n    def check_auth(self):\n        """Validate Authorization: Bearer <API_KEY> header."""\n        if config is None or not config.api_key:\n            self.send_error_response(503, "Supervisor authentication is not configured")\n            return False\n        auth_header = self.headers.get("Authorization", "")\n        if not auth_header.startswith("Bearer "):\n            self.send_error_response(401, "Missing or invalid Authorization header. Expected Bearer token.")\n            return False\n        token = auth_header[7:].strip()\n        if not hmac.compare_digest(token.encode(), config.api_key.encode()):\n            self.send_error_response(403, "Forbidden: Invalid API Key.")\n            return False\n        return True\n\n    def send_error_response(self, status_code, message):\n        body = json.dumps({"error": {"message": scrub(message), "code": status_code}}).encode("utf-8")\n        self.send_response(status_code)\n        self.send_header("Content-Type", "application/json")\n        self.send_header("Content-Length", str(len(body)))\n        self.end_headers()\n        self.wfile.write(body)\n\n    def do_GET(self):\n        if not self.check_auth():\n            return\n        parsed = urllib.parse.urlparse(self.path)\n        path = parsed.path\n\n        if path == "/health":\n            llama_ok = check_llama_health()\n            container_uptime = time.time() - config.start_time\n            gpus = get_gpu_telemetry()\n            payload = {\n                "status": "healthy" if llama_ok else "degraded",\n                "supervisorUptimeSeconds": round(time.time() - config.start_time, 1),\n                "sessionAgeSeconds": round(container_uptime, 1),\n                "sessionAgeSource": "supervisor_start_estimate",\n                "isEstimate": True,\n                "linuxUptimeSeconds": round(get_container_uptime_seconds(), 1),\n                "maxSessionSeconds": 43200,\n                "secondsRemainingIn12hSession": max(0, round(43200 - container_uptime, 1)),\n                "llamaServer": {\n                    "healthy": llama_ok,\n                    "port": config.llama_port,\n                    "pid": config.llama_process.pid if config.llama_process else None,\n                },\n                "gpus": gpus,\n            }\n            body = json.dumps(payload, indent=2).encode("utf-8")\n            self.send_response(200 if llama_ok else 503)\n            self.send_header("Content-Type", "application/json")\n            self.send_header("Content-Length", str(len(body)))\n            self.end_headers()\n            self.wfile.write(body)\n            return\n\n        if not self.check_auth():\n            return\n\n        if path == "/v1/models" and not parsed.query:\n            self.proxy_to_llama("GET")\n        else:\n            self.send_error_response(404, f"Endpoint {path} not found.")\n\n    def do_POST(self):\n        parsed = urllib.parse.urlparse(self.path)\n        path = parsed.path\n\n        if not self.check_auth():\n            return\n\n        if path == "/control/restart" and not parsed.query:\n            if not config.llama_cmd:\n                self.send_error_response(409, "Supervisor does not own a restart command")\n                return\n            if config.llama_process and config.llama_process.poll() is None:\n                config.llama_process.terminate()\n                try:\n                    config.llama_process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    config.llama_process.kill()\n            start_llama_server()\n            body = json.dumps({"status": "restarting", "message": "llama-server restart signal sent."}).encode("utf-8")\n            self.send_response(200)\n            self.send_header("Content-Type", "application/json")\n            self.send_header("Content-Length", str(len(body)))\n            self.end_headers()\n            self.wfile.write(body)\n            return\n\n        if path == "/v1/chat/completions" and not parsed.query:\n            self.proxy_to_llama("POST")\n        else:\n            self.send_error_response(404, f"Endpoint {path} not found.")\n\n    def proxy_to_llama(self, method):\n        target_url = f"http://127.0.0.1:{config.llama_port}{self.path}"\n        try:\n            content_length = int(self.headers.get("Content-Length", 0))\n            if not 0 <= content_length <= 16 * 1024 * 1024 or self.headers.get("Transfer-Encoding"):\n                raise ValueError()\n        except ValueError:\n            self.send_error_response(400, "Invalid request body length")\n            return\n        body = self.rfile.read(content_length) if content_length > 0 else None\n\n        headers = {}\n        for k, v in self.headers.items():\n            if k.lower() not in ("host", "authorization", "content-length"):\n                headers[k] = v\n        if content_length > 0:\n            headers["Content-Length"] = str(content_length)\n\n        req = urllib.request.Request(target_url, data=body, headers=headers, method=method)\n\n        try:\n            with urllib.request.urlopen(req, timeout=900) as resp:\n                self.send_response(resp.status)\n                is_sse = False\n                for k, v in resp.headers.items():\n                    if k.lower() == "transfer-encoding" and v.lower() == "chunked":\n                        continue\n                    if k.lower() == "content-type" and "text/event-stream" in v.lower():\n                        is_sse = True\n                    self.send_header(k, v)\n                self.send_header("Connection", "close")\n                self.close_connection = True\n                self.end_headers()\n\n                while True:\n                    chunk = resp.readline() if is_sse else resp.read(8192)\n                    if not chunk:\n                        break\n                    try:\n                        self.wfile.write(chunk)\n                        self.wfile.flush()\n                    except (BrokenPipeError, ConnectionResetError):\n                        break\n        except urllib.error.HTTPError as err:\n            self.send_error_response(err.code, "Internal inference backend returned an error")\n        except Exception:\n            self.send_error_response(502, "Internal inference backend unavailable")\n\n\nclass ThreadedHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):\n    daemon_threads = True\n    allow_reuse_address = True\n\n\ndef run_supervisor(port=8081, llama_port=8080, api_key=None, llama_cmd=None, log_path=""):\n    global config\n    config = SupervisorConfig(port, llama_port, api_key, llama_cmd, log_path)\n\n    if config.llama_cmd:\n        start_llama_server()\n\n    server_address = ("0.0.0.0", config.port)\n    httpd = ThreadedHTTPServer(server_address, SupervisorHandler)\n    try:\n        httpd.serve_forever()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        httpd.server_close()\n        if config.llama_process and config.llama_process.poll() is None:\n            config.llama_process.terminate()\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Kaggle llama-server Authenticated Supervisor")\n    parser.add_argument("--port", type=int, default=8081, help="Supervisor external port (default: 8081)")\n    parser.add_argument("--llama-port", type=int, default=8080, help="Internal llama-server port (default: 8080)")\n    parser.add_argument("--api-key", type=str, default=None, help="Bearer token for API access")\n    parser.add_argument("--log-path", type=str, default="/kaggle/working/llama_server.log", help="Log output file path")\n    args = parser.parse_args()\n\n    run_supervisor(port=args.port, llama_port=args.llama_port, api_key=args.api_key, log_path=args.log_path)\n'
SUPERVISOR_PY.write_text(supervisor_script, encoding='utf-8')

# Start llama-server
llama_cmd = [
    str(SERVER_BIN),
    '--model', str(MODEL_PATH),
    '--alias', CONFIG['MODEL_ALIAS'],
    '--host', '127.0.0.1',
    '--port', str(CONFIG['LLAMA_PORT']),
    '--n-gpu-layers', str(CONFIG['N_GPU_LAYERS']),
    '--split-mode', CONFIG['SPLIT_MODE'],
    '--tensor-split', CONFIG['TENSOR_SPLIT'],
    '--ctx-size', str(CONFIG['CTX_SIZE']),
    '--parallel', '1',
    '--fit', 'off',
    '--batch-size', str(CONFIG['BATCH_SIZE']),
    '--ubatch-size', str(CONFIG['UBATCH_SIZE']),
    '--cache-type-k', CONFIG['CACHE_TYPE_K'],
    '--cache-type-v', CONFIG['CACHE_TYPE_V'],
    '--flash-attn', 'auto',
    '--jinja',
    '--no-context-shift',
]

llama_env = os.environ.copy()
if SERVER_BIN:
    lib_dir = str(Path(SERVER_BIN).parent)
    llama_env['LD_LIBRARY_PATH'] = f"{lib_dir}:{llama_env.get('LD_LIBRARY_PATH', '')}"

with open(LOG_PATH, 'w', encoding='utf-8') as f:
    llama_proc = subprocess.Popen(llama_cmd, stdout=f, stderr=subprocess.STDOUT, env=llama_env)

print('Booting llama-server...')
time.sleep(5)

# Start supervisor
env = os.environ.copy()
env['SUPERVISOR_API_KEY'] = CONFIG['API_KEY']
env['LLAMA_PORT'] = str(CONFIG['LLAMA_PORT'])
supervisor_proc = subprocess.Popen(
    [sys.executable, str(SUPERVISOR_PY), '--port', str(CONFIG['SUPERVISOR_PORT']), '--llama-port', str(CONFIG['LLAMA_PORT'])],
    env=env
)

print(f'Waiting for llama-server model load on port {CONFIG["LLAMA_PORT"]} (up to 8 mins)...')
healthy = False
for i in range(96):
    try:
        r = requests.get(f'http://127.0.0.1:{CONFIG["SUPERVISOR_PORT"]}/health', headers={'Authorization': 'Bearer ' + CONFIG['API_KEY']}, timeout=2)
        if r.status_code == 200 and r.json().get('llamaServer', {}).get('healthy'):
            healthy = True
            break
    except Exception:
        pass
    if i % 6 == 0:
        print(f'Loading model layers onto dual T4 GPUs... {(i + 1) * 5}s')
    time.sleep(5)

if not healthy:
    print('Failed to start. Server log tail:')
    subprocess.run(['tail', '-n', '50', str(LOG_PATH)], check=False)
    raise RuntimeError('Model failed to become healthy.')

print('*** SUPERVISOR HEALTHY ON PORT', CONFIG['SUPERVISOR_PORT'], '***')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total', '--format=csv,noheader'], check=True)

In [ ]:
# 7. NETWORK BRIDGE - Cloudflare URL or Tailscale TCP
import os, shutil, subprocess, time, ipaddress, re, queue, threading
from pathlib import Path

if supervisor_proc.poll() is not None or llama_proc.poll() is not None:
    raise RuntimeError('Start and verify the worker in code cell 6 first.')
if any(globals().get(name) is not None and globals()[name].poll() is None
       for name in ('ts_daemon', 'cf_proc')):
    raise RuntimeError('A transport is already running; do not rerun this cell.')

if CONFIG['TRANSPORT'] == 'cloudflare':
    cf_bin = SCRATCH / 'cloudflared'
    if not cf_bin.is_file():
        subprocess.run(['curl', '-fsSL',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-o', str(cf_bin)], check=True)
        cf_bin.chmod(0o755)
    cf_proc = subprocess.Popen([str(cf_bin), 'tunnel', '--protocol', 'http2',
        '--no-autoupdate', '--url', f'http://127.0.0.1:{CONFIG["SUPERVISOR_PORT"]}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    url_queue = queue.Queue(maxsize=1)
    def read_tunnel_output():
        # Consume diagnostics without printing or persisting them.
        for line in cf_proc.stdout:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match and url_queue.empty():
                url_queue.put_nowait(match.group(0))
    threading.Thread(target=read_tunnel_output, daemon=True).start()
    try:
        remote_url = url_queue.get(timeout=60)
    except queue.Empty:
        if cf_proc.poll() is None:
            cf_proc.terminate()
        raise RuntimeError('Cloudflare tunnel did not produce a URL in 60 seconds.') from None
    # A URL alone is not the bearer credential. Show it to the notebook owner once.
    scrubber.DEFAULT_PATTERNS = [item for item in scrubber.DEFAULT_PATTERNS
                                 if 'trycloudflare' not in item[0].pattern]
    print('CLOUDFLARE QUICK TUNNEL ONLINE (streaming/SSE unsupported).')
else:

    TS_DIR = SCRATCH / 'tailscale'
    TS_DIR.mkdir(parents=True, exist_ok=True)
    TSD, TS_CLI = TS_DIR / 'tailscaled', TS_DIR / 'tailscale'
    if not TSD.is_file() or not TS_CLI.is_file():
        subprocess.run(['curl', '-fsSL',
            'https://pkgs.tailscale.com/stable/tailscale_1.76.6_amd64.tgz',
            '-o', str(SCRATCH / 'tailscale.tgz')], check=True)
        subprocess.run(['tar', '-xzf', str(SCRATCH / 'tailscale.tgz'),
            '--strip-components=1', '-C', str(TS_DIR)], check=True)

    # Capture and scrub command diagnostics; never include a key in argv or output.
    def ts_run(args):
        result = subprocess.run([str(TS_CLI), f'--socket={SCRATCH}/tailscaled.sock', *args],
                                capture_output=True, text=True, timeout=90)
        if result.returncode:
            raise RuntimeError(scrubber.scrub(result.stderr or 'Tailscale command failed.'))
        return result.stdout.strip()

    ts_daemon = subprocess.Popen([str(TSD), '--tun=userspace-networking',
        f'--state={SCRATCH}/tailscaled.state', f'--socket={SCRATCH}/tailscaled.sock'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        for _ in range(30):
            if ts_daemon.poll() is not None:
                raise RuntimeError('Tailscale daemon exited.')
            if (SCRATCH / 'tailscaled.sock').exists():
                break
            time.sleep(1)
        else:
            raise RuntimeError('Tailscale socket did not appear.')

        auth_file = SCRATCH / 'tailscale-auth.key'
        fd = os.open(auth_file, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        try:
            with os.fdopen(fd, 'w', encoding='utf-8') as handle:
                handle.write(CONFIG['TAILSCALE_AUTHKEY'])
            ts_run(['up', f'--auth-key=file:{auth_file}',
                    '--hostname=kaggle-freecompute-brain', '--timeout=60s'])
        finally:
            auth_file.unlink(missing_ok=True)

        status = str(ipaddress.IPv4Address(ts_run(['ip', '-4'])))
        # Userspace networking needs an explicit inbound listener. Raw TCP preserves SSE.
        ts_run(['serve', '--bg', f'--tcp={CONFIG["SUPERVISOR_PORT"]}',
                f'tcp://127.0.0.1:{CONFIG["SUPERVISOR_PORT"]}'])
    except Exception:
        ts_daemon.terminate()
        ts_daemon.wait(timeout=10)
        raise

    remote_url = f'http://{status}:{CONFIG["SUPERVISOR_PORT"]}'
    print('TAILSCALE TCP FORWARDING CONFIGURED (local connectivity still unverified).')

print('Remote URL:', remote_url)
print('Bearer token: use the same FREECOMPUTE_API_KEY secret locally; it is not printed.')
print('Next: run code cell 8, then return its manifest and this URL to Codex.')


In [ ]:
# 8. AUTHENTICATED READINESS + MANIFEST ? no inference outside the local Core
import json

base_url = f'http://127.0.0.1:{CONFIG["SUPERVISOR_PORT"]}'
headers = {'Authorization': 'Bearer ' + CONFIG['API_KEY']}
health_res = requests.get(base_url + '/health', headers=headers, timeout=10)
health_res.raise_for_status()
health = health_res.json()
if health.get('status') != 'healthy' or not health.get('llamaServer', {}).get('healthy'):
    raise RuntimeError('Supervisor/model not healthy; stop before acceptance.')
models_res = requests.get(base_url + '/v1/models', headers=headers, timeout=10)
models_res.raise_for_status()
model_ids = [model['id'] for model in models_res.json().get('data', [])]
if CONFIG['MODEL_ALIAS'] not in model_ids:
    raise RuntimeError('Expected model alias is not advertised.')
manifest = {
    'worker_id': 'kaggle-qwen', 'profile_id': 'kaggle-qwen-historical-64k',
    'location': 'kaggle', 'engine': 'llama.cpp',
    'llama_version_observed': subprocess.check_output([str(SERVER_BIN), '--version'],
                                                    text=True, stderr=subprocess.STDOUT).strip(),
    'llama_commit_requested': CONFIG['LLAMA_COMMIT'],
    'model_repo': CONFIG['REPO_ID'], 'model_filename': CONFIG['FILENAME'],
    'model_revision_requested': CONFIG['REVISION'], 'model_bytes_observed': MODEL_PATH.stat().st_size,
    'quantization_filename': 'UD-DW-Q4_K_M', 'context_requested': CONFIG['CTX_SIZE'],
    'tokenizer_template': 'GGUF embedded; --jinja; identities not yet freshly verified',
    'concurrency': 1, 'resource_pool': 'kaggle-dual-t4', 'resources': ['gpu0', 'gpu1'],
    'auth': 'bearer', 'transport': CONFIG['TRANSPORT'], 'engine_argv': llama_cmd,
    'gpu_observations': health.get('gpus', []), 'advertised_models': model_ids,
    'verification': 'startup observed; inference/streaming/tools/context UNVERIFIED',
}
print(json.dumps(scrubber.structured(manifest), indent=2))
print('WORKER READY FOR LOCAL ACCEPTANCE')


In [ ]:
# 9. SHUTDOWN ? only after local acceptance has finished
CONFIRM_SHUTDOWN = False  # Change to True only when finished; do not Run All.
if CONFIRM_SHUTDOWN:
    for name in ('cf_proc', 'ts_daemon', 'supervisor_proc', 'llama_proc'):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=15)
    print('Worker stopped. Click Stop Session in Kaggle to end GPU allocation.')
else:
    print('Shutdown skipped. Leave the worker running for local acceptance.')
